# 02 — Features and leakage audit

Build `pre_tournament` feature rows at each match's `prediction_cutoff` and persist to the frozen feature store.
See `docs/leakage_audit.md` for regime rules.

In [10]:
from tml.features.builders import build_feature_row, prediction_cutoff
from tml.features.elo import EloState
from tml.features.store import FeatureStore

In [11]:
# Load modeling table from notebook 01 (or rebuild via tml.data).
# modeling: pd.DataFrame with player_a_id, player_b_id, tourney_date, ...
store = FeatureStore()
elo = EloState()

In [12]:
from pathlib import Path
import pandas as pd

modeling = pd.read_parquet(Path("../data/processed/modeling.parquet"))
modeling.head()

,match_id,tourney_id,tourney_date,surface,best_of,tour_level,player_a_id,player_b_id,y_complete_win,prediction_regime,...,age_a,age_b,a_svpt,b_svpt,a_1stIn,b_1stIn,a_1stWon,b_1stWon,a_2ndWon,b_2ndWon
0,atp:2000-7308:1,2000-7308,2000-01-03,Hard,3,atp,C487,E113,0,pre_tournament,...,22.045,25.810,59.0,66.0,37.0,29.0,25.0,23.0,13.0,23.0
1,atp:2000-7308:2,2000-7308,2000-01-03,Hard,3,atp,F324,K260,1,pre_tournament,...,18.404,24.882,46.0,42.0,28.0,15.0,24.0,13.0,12.0,12.0
2,atp:2000-7308:3,2000-7308,2000-01-03,Hard,3,atp,A202,G352,0,pre_tournament,...,28.797,22.585,103.0,81.0,59.0,40.0,49.0,35.0,22.0,28.0
3,atp:2000-7308:4,2000-7308,2000-01-03,Hard,3,atp,G379,I052,1,pre_tournament,...,21.599,23.710,66.0,49.0,35.0,22.0,28.0,12.0,14.0,8.0
4,atp:2000-7308:5,2000-7308,2000-01-03,Hard,3,atp,D270,N250,0,pre_tournament,...,25.580,23.595,73.0,52.0,40.0,32.0,25.0,26.0,16.0,12.0


In [14]:
# Example: show cutoff semantics for one tournament row.
cutoff = prediction_cutoff(modeling.iloc[0]["tourney_date"])

match = modeling.iloc[0].copy()
match["dataset_snapshot_id"] = match.get("dataset_snapshot_id") or "notebook-manual"

row = build_feature_row(match, elo, pd.DataFrame())  # or history=[]
store.persist([row])

PosixPath('data/processed/features/features.parquet')

In [15]:
# QC: loaded rows must all be pre_tournament with unique match_id.
features = store.load()
assert features["prediction_regime"].eq("pre_tournament").all()